# K03_00 – Datenrepräsentation und Vorverarbeitung – Studentenversion

Diese Fassung ist für die **aktive Mitarbeit im Kurs** gedacht.

Dieses Notebook ist der **Einstieg in Kapitel 3**.  
Wir schauen uns an, wie tabellarische Daten im Machine Learning dargestellt werden und warum Vorverarbeitung oft notwendig ist.

## Lernziele
Nach diesem Notebook können Sie:
- zwischen **Samples**, **Features** und **Label** unterscheiden
- numerische und kategoriale Merkmale erkennen
- **fehlende Werte** identifizieren
- einfache **One-Hot-Kodierung** anwenden
- den Effekt von **Skalierung** verstehen

## 1. Ein kleines Beispieldataset

Wir verwenden einen kleinen fiktiven Datensatz zu Wohnungsangeboten.  
Jede **Zeile** ist ein Sample, jede **Spalte** ein Merkmal.

Die Zielvariable heißt hier `preis_in_tsd`.


In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "wohnflaeche": [45, 52, 68, 80, 95, 110, 130, 70],
    "zimmer": [2, 2, 3, 3, 4, 4, 5, 3],
    "lage": ["einfach", "mittel", "mittel", "gut", "gut", "gut", "sehr_gut", "mittel"],
    "baujahr": [1970, 1985, 1995, 2001, np.nan, 2010, 2018, 1998],
    "preis_in_tsd": [120, 145, 180, 220, 260, 310, 420, 195]
})

df


,wohnflaeche,zimmer,lage,baujahr,preis_in_tsd
0,45,2,einfach,1970.0,120
1,52,2,mittel,1985.0,145
2,68,3,mittel,1995.0,180
3,80,3,gut,2001.0,220
4,95,4,gut,NaN,260
5,110,4,gut,2010.0,310
6,130,5,sehr_gut,2018.0,420
7,70,3,mittel,1998.0,195


### Merksatz
- **Sample** = eine Beobachtung, also eine Zeile
- **Features** = beschreibende Merkmale
- **Label** = Zielwert, den wir vorhersagen wollen


In [2]:
X = df.drop(columns="preis_in_tsd")
y = df["preis_in_tsd"]

print("Feature-Matrix X:")
display(X)

print("Zielvariable y:")
display(y)


Feature-Matrix X:


,wohnflaeche,zimmer,lage,baujahr
0,45,2,einfach,1970.0
1,52,2,mittel,1985.0
2,68,3,mittel,1995.0
3,80,3,gut,2001.0
4,95,4,gut,NaN
5,110,4,gut,2010.0
6,130,5,sehr_gut,2018.0
7,70,3,mittel,1998.0


Zielvariable y:


,preis_in_tsd
0,120
1,145
2,180
3,220
4,260
5,310
6,420
7,195


## Mini-Übung 1
Beantworten Sie kurz:
1. Wie viele Samples enthält der Datensatz?
2. Wie viele Features hat `X`?
3. Welche Spalte ist das Label?


In [3]:
print("Anzahl Samples:", X.shape[0])
print("Anzahl Features:", X.shape[1])
print("Label-Spalte:", "preis_in_tsd")


Anzahl Samples: 8
Anzahl Features: 4
Label-Spalte: preis_in_tsd


## 2. Numerische und kategoriale Merkmale

Nicht alle Spalten haben denselben Typ:
- `wohnflaeche`, `zimmer`, `baujahr` sind **numerisch**
- `lage` ist **kategorial**

Viele klassische ML-Modelle erwarten am Ende **numerische Eingaben**.


In [4]:
print(df.dtypes)


wohnflaeche       int64
zimmer            int64
lage             object
baujahr         float64
preis_in_tsd      int64
dtype: object


## 3. Fehlende Werte

In realen Daten sind Einträge oft unvollständig.  
Hier fehlt ein Wert in der Spalte `baujahr`.


In [5]:
print(df.isna().sum())


wohnflaeche     0
zimmer          0
lage            0
baujahr         1
preis_in_tsd    0
dtype: int64


### Mini-Übung 2
Welche Strategien wären hier denkbar?
- Zeile löschen
- Spalte löschen
- Wert schätzen (imputieren)

Im nächsten Schritt verwenden wir eine einfache Schätzung mit dem Median.


In [6]:
df_imputed = df.copy()
median_baujahr = df_imputed["baujahr"].median()
df_imputed["baujahr"] = df_imputed["baujahr"].fillna(median_baujahr)

print("Verwendeter Median:", median_baujahr)
df_imputed


Verwendeter Median: 1998.0


,wohnflaeche,zimmer,lage,baujahr,preis_in_tsd
0,45,2,einfach,1970.0,120
1,52,2,mittel,1985.0,145
2,68,3,mittel,1995.0,180
3,80,3,gut,2001.0,220
4,95,4,gut,1998.0,260
5,110,4,gut,2010.0,310
6,130,5,sehr_gut,2018.0,420
7,70,3,mittel,1998.0,195


## 4. Kategoriale Merkmale kodieren

Die Spalte `lage` ist Text.  
Damit ein Modell später damit arbeiten kann, kodieren wir sie mit **One-Hot-Encoding**.


In [7]:
df_encoded = pd.get_dummies(df_imputed, columns=["lage"], dtype=int)
df_encoded


,wohnflaeche,zimmer,baujahr,preis_in_tsd,lage_einfach,lage_gut,lage_mittel,lage_sehr_gut
0,45,2,1970.0,120,1,0,0,0
1,52,2,1985.0,145,0,0,1,0
2,68,3,1995.0,180,0,0,1,0
3,80,3,2001.0,220,0,1,0,0
4,95,4,1998.0,260,0,1,0,0
5,110,4,2010.0,310,0,1,0,0
6,130,5,2018.0,420,0,0,0,1
7,70,3,1998.0,195,0,0,1,0


### Beobachtung
Aus **einer** kategorialen Spalte entstehen **mehrere** numerische Spalten.


## 5. Warum ist Skalierung wichtig?

Betrachten wir zwei Merkmale:
- `wohnflaeche` liegt ungefähr zwischen 45 und 130
- `zimmer` liegt nur zwischen 2 und 5

Ohne Skalierung hat `wohnflaeche` numerisch eine viel größere Größenordnung.
Das kann für distanzbasierte Verfahren problematisch sein.


In [8]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

features_numeric = df_imputed[["wohnflaeche", "zimmer", "baujahr"]]

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

X_standard = pd.DataFrame(
    standard_scaler.fit_transform(features_numeric),
    columns=features_numeric.columns
)

X_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(features_numeric),
    columns=features_numeric.columns
)

print("Originaldaten:")
display(features_numeric.head())

print("Standardisiert:")
display(X_standard.head())

print("Min-Max-skaliert:")
display(X_minmax.head())


Originaldaten:


,wohnflaeche,zimmer,baujahr
0,45,2,1970.0
1,52,2,1985.0
2,68,3,1995.0
3,80,3,2001.0
4,95,4,1998.0


Standardisiert:


,wohnflaeche,zimmer,baujahr
0,-1.338753,-1.290994,-1.959491
1,-1.080235,-1.290994,-0.865822
2,-0.489337,-0.258199,-0.136709
3,-0.046164,-0.258199,0.300759
4,0.507803,0.774597,0.082025


Min-Max-skaliert:


,wohnflaeche,zimmer,baujahr
0,0.000000,0.000000,0.000000
1,0.082353,0.000000,0.312500
2,0.270588,0.333333,0.520833
3,0.411765,0.333333,0.645833
4,0.588235,0.666667,0.583333


## Mini-Übung 3
Prüfen Sie:
1. Welchen Mittelwert hat jede Spalte nach der Standardisierung ungefähr?
2. In welchem Bereich liegen die Werte nach der Min-Max-Skalierung?


In [9]:
print("Mittelwerte nach Standardisierung:")
print(X_standard.mean().round(6))

print("\nMinima nach Min-Max-Skalierung:")
print(X_minmax.min().round(6))

print("\nMaxima nach Min-Max-Skalierung:")
print(X_minmax.max().round(6))


Mittelwerte nach Standardisierung:
wohnflaeche    0.0
zimmer         0.0
baujahr        0.0
dtype: float64

Minima nach Min-Max-Skalierung:
wohnflaeche    0.0
zimmer         0.0
baujahr        0.0
dtype: float64

Maxima nach Min-Max-Skalierung:
wohnflaeche    1.0
zimmer         1.0
baujahr        1.0
dtype: float64


## 6. Kleine Zusammenfassung

In diesem Notebook haben wir gesehen:
- tabellarische Daten werden in **X** und **y** getrennt
- Merkmale können unterschiedliche Typen haben
- fehlende Werte müssen behandelt werden
- kategoriale Merkmale müssen oft kodiert werden
- Skalierung ist für viele Verfahren methodisch wichtig

## Transferfrage
Warum wäre es problematisch, Textspalten oder fehlende Werte einfach ungeprüft an ein Modell weiterzugeben?

Bei Textspalten kann es passieren, dass der ML -Algorithmus nicht damit umgehen kann, da die meisten Algorithmen nur mit numerischen Werten arbeiten können
Bei Fehlenden Werten kann es zu Verzerrungen und Fehlern beim Ergebnis kommen.



## Was bringt die Min-Max-Skalierung?

Min-Max-Skalierung, auch als Normalisierung bekannt, skaliert (oder verschiebt und skaliert) jeden Wert im Feature-Bereich so, dass er im Bereich von 0 und 1 (oder einem benutzerdefinierten Bereich) liegt.

Die Formel dafür ist:

```
X_scaled = (X - X_min) / (X_max - X_min)
```

**Warum ist das nützlich?**

1.  **Einheitlicher Bereich:** Alle Merkmale liegen im gleichen Wertebereich. Dies verhindert, dass Merkmale mit größeren ursprünglichen Werten (wie `wohnflaeche`) die Analyse oder das Training eines Modells dominieren, im Vergleich zu Merkmalen mit kleineren Werten (wie `zimmer`).
2.  **Vorteilhaft für bestimmte Algorithmen:** Viele Machine-Learning-Algorithmen (z. B. neuronale Netze, Support Vector Machines, K-Means Clustering) funktionieren besser oder konvergieren schneller, wenn die Eingabemerkmale skaliert sind. Dies liegt daran, dass sie oft auf Distanzberechnungen basieren oder anfällig für unterschiedliche Skalen sind.
3.  **Visualisierung:** Skalierte Daten können auch die Visualisierung erleichtern.

Wie Sie in der `X_minmax` Ausgabe gesehen haben, liegen alle Werte für `wohnflaeche`, `zimmer` und `baujahr` jetzt zwischen 0 und 1.